# Análisis Comparativo: Monotributo vs. Responsable Inscripto
## Costo Fiscal Total — Argentina 2026 (1er Semestre)

### Objetivo
Comparar el costo fiscal real de pertenecer al régimen de **Monotributo** frente al régimen de **Responsable Inscripto (RI)** para dos escenarios de facturación:
- **Escenario H**: Máxima facturación posible dentro de la Categoría H del Monotributo
- **Escenario K**: Máxima facturación posible dentro de la Categoría K del Monotributo

### Supuestos generales
- Actividad: **prestación de servicios** (no venta de cosas muebles)
- Facturación al **máximo permitido** en cada categoría
- Clientes **Responsables Inscriptos** (IVA neutro para el prestador)
- Profesional **soltero, sin cargas de familia**
- **Se excluye** del análisis el costo de obra social (puede redirigirse a prepaga)
- Costos profesionales deducibles en RI: **20% de la facturación bruta**
- Honorarios contador en Monotributo: **$55.000/mes**
- Honorarios contador en RI: estimado según complejidad del régimen

### Fuentes
- [ARCA (ex AFIP) — Categorías Monotributo](https://www.afip.gob.ar/monotributo/categorias.asp)
- [Contablix — Tabla completa Feb 2026](https://contablix.ar/monotributo-categorias-2026/)
- [ARCA — Autónomos categorías y aportes](https://www.arca.gob.ar/autonomos/categorias-y-aportes/2026.asp)
- [Ignacio Online — Autónomos 2026 valores vigentes desde enero](https://www.ignacioonline.com.ar/autonomos-2026-valores-vigentes-desde-enero/)
- [Contablix — Responsable Inscripto guía 2026](https://contablix.ar/blog/responsable-inscripto-guia-definitiva-impuestos/)
- [iProfesional — Ganancias 2026 escala y deducciones](https://www.iprofesional.com/impuestos/446080-ganancias-2026-nuevos-valores-de-deducciones-escala-y-piso-salarial)
- [Jorge Vega — Autónomos enero 2026](https://jorgevega.com.ar/laboral/419-autonomos-aportes-categorias-2026-enero.html)
- [Tributosimple — Monotributo vigente](https://tributosimple.com/categorias-del-monotributo-vigentes/)

> **Vigencia:** Valores del 1er semestre 2026 (actualización feb 2026, variación IPC 2do semestre 2025: +14,29%)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

pd.set_option('display.float_format', lambda x: f'{x:,.0f}')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# ─────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────

def fmt_pesos(v):
    """Format a number as Argentine pesos."""
    return f'${v:>15,.0f}'

def fmt_pct(v):
    return f'{v:.2f}%'

print('Librerías cargadas correctamente.')

---
## 1. Parámetros del Monotributo (Feb 2026, servicios)

### Estructura de la cuota mensual
La cuota del monotributo se compone de tres partes:
1. **Impuesto integrado** — reemplaza IVA + Ganancias
2. **Aporte SIPA** — jubilación/previsional
3. **Aporte obra social** — *(excluido del análisis)*

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# MONOTRIBUTO — Parámetros (Feb 2026, prestación de servicios)
# Fuente: ARCA / Contablix tabla completa Feb 2026
# ─────────────────────────────────────────────────────────────────────

MONO = {
    'H': {
        'billing_anual_max':      70_113_407,   # Límite superior de la categoría
        'impuesto_integrado_mes': 350_567,       # Reemplaza IVA + Ganancias
        'sipa_mes':                49_294,       # Aporte jubilatorio (SIPA)
        'obra_social_mes':         47_486,       # Obra social (EXCLUIDA del análisis)
        'total_cuota_mes':        447_347,       # Total mensual (verificación: suma de los 3 componentes)
    },
    'K': {
        'billing_anual_max':     108_357_084,
        'impuesto_integrado_mes': 1_171_213,
        'sipa_mes':                135_263,
        'obra_social_mes':          75_213,
        'total_cuota_mes':        1_381_689,
    }
}

# Honorarios del contador para Monotributo (dato del enunciado)
MONO_CONTADOR_MES = 55_000

# ── Verificación de integridad de datos ──
for cat, d in MONO.items():
    suma = d['impuesto_integrado_mes'] + d['sipa_mes'] + d['obra_social_mes']
    diff = abs(suma - d['total_cuota_mes'])
    status = '✓' if diff <= 5 else f'⚠ dif={diff:,.0f}'
    print(f"Cat {cat}: impuesto+SIPA+OS = {fmt_pesos(suma)} | total declarado = {fmt_pesos(d['total_cuota_mes'])} | {status}")

print(f"\nHonorarios contador Monotributo: {fmt_pesos(MONO_CONTADOR_MES)}/mes")

In [ ]:
# ── Tabla resumen Monotributo ──
rows = []
for cat in ['H', 'K']:
    d = MONO[cat]
    rows.append({
        'Categoría': cat,
        'Billing máx. anual (ARS)': d['billing_anual_max'],
        'Billing máx. mensual (ARS)': d['billing_anual_max'] / 12,
        'Impuesto integrado / mes': d['impuesto_integrado_mes'],
        'SIPA jubilación / mes': d['sipa_mes'],
        'Obra social / mes (excluida)': d['obra_social_mes'],
        'Cuota total / mes': d['total_cuota_mes'],
    })

df_mono = pd.DataFrame(rows).set_index('Categoría')
print('=== TABLA DE CUOTAS MONOTRIBUTO (mensual, servicios, Feb 2026) ===')
display(df_mono)

---
## 2. Parámetros del Régimen de Responsable Inscripto

### Componentes del costo fiscal en RI
1. **IVA (21%)**: Neutral para B2B — el cliente RI acredita el IVA facturado. No es un costo real del prestador.
2. **Ganancias**: Impuesto progresivo (5% al 35%) sobre la ganancia neta imponible.
3. **Autónomos (SIPA)**: Aporte mensual fijo por categoría de ingresos.
4. **Honorarios contador**: Mayor complejidad que en monotributo (DDJJ IVA mensual, ganancias, autónomos).

### Autónomos — Categoría aplicable
Las categorías de autónomos se determinan por la renta mensual de referencia (estimada). Un profesional facturando $5-9M/mes se encuentra en la **Categoría V** (la máxima), con un aporte mensual de **$276.065,92**.

La contribución autónomos se destina a:
- **27% de la base de referencia → SIPA (jubilación)**
- **5% → INSSJP (PAMI)**
- **6% → Obra social**
- Total: 38% de la base notional

> Por las instrucciones del análisis, se excluyen PAMI y obra social, contabilizando únicamente el aporte jubilatorio (SIPA = 27/38 = 71,1% del total).

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# AUTÓNOMOS — Parámetros (Enero 2026, Res. ANSES 381/2025)
# Fuente: ARCA / IgnacioOnline / JorgeVega
# ─────────────────────────────────────────────────────────────────────

# Tabla de categorías de autónomos — Enero 2026
# Categoría determinada por renta mensual de referencia
autonomos_tabla = [
    {'Categoría': 'I',   'Renta ref. mensual':   196_072.13, 'Aporte mensual total':  62_743.08},
    {'Categoría': 'II',  'Renta ref. mensual':   274_496.78, 'Aporte mensual total':  87_838.96},
    {'Categoría': 'III', 'Renta ref. mensual':   392_140.83, 'Aporte mensual total': 125_485.06},
    {'Categoría': 'IV',  'Renta ref. mensual':   627_425.04, 'Aporte mensual total': 200_776.01},
    {'Categoría': 'V',   'Renta ref. mensual':   862_706.01, 'Aporte mensual total': 276_065.92},  # ← MÁXIMA
]

df_autonomos = pd.DataFrame(autonomos_tabla).set_index('Categoría')
print('=== TABLA AUTÓNOMOS — Aportes mensuales (Enero 2026) ===')
display(df_autonomos)

# Para los escenarios H y K, la renta mensual del profesional supera
# ampliamente el umbral de Cat V → se aplica Cat V
AUTONOMOS_CAT_V_MES = 276_065.92

# Desglose porcentual del aporte Cat V:
# Tasa SIPA (jubilación): 27%
# Tasa PAMI:               5%
# Tasa obra social:         6%
# Total:                   38%  →  base notional = aporte / 0.38

FRAC_SIPA = 27 / 38   # porción jubilación del total
FRAC_SALUD = 11 / 38  # porción salud (PAMI + OS) del total

autonomos_sipa_mes   = AUTONOMOS_CAT_V_MES * FRAC_SIPA   # ← costo previsional (INCLUIDO)
autonomos_salud_mes  = AUTONOMOS_CAT_V_MES * FRAC_SALUD  # ← salud (EXCLUIDA)

print(f"\nAporte Cat V total:          {fmt_pesos(AUTONOMOS_CAT_V_MES)}/mes")
print(f"  └─ SIPA jubilación (71,1%): {fmt_pesos(autonomos_sipa_mes)}/mes  ← incluido en el análisis")
print(f"  └─ PAMI + OS  (28,9%):      {fmt_pesos(autonomos_salud_mes)}/mes  ← excluido del análisis")

### Escala del Impuesto a las Ganancias — 1er Semestre 2026

Sistema de alícuotas progresivas (Art. 94 LIG), actualizado +14,29% por IPC 2do semestre 2025.

| Ganancia Neta Imponible acumulada (anual) | Importe fijo | Alícuota s/ excedente |
|---|---|---|
| Hasta $2.000.030 | $0 | 5% |
| $2.000.030 – $4.000.060 | $100.002 | 9% |
| $4.000.060 – $6.000.090 | $280.004 | 12% |
| $6.000.090 – $9.000.135 | $520.008 | 15% |
| $9.000.135 – $18.000.271 | $970.015 | 19% |
| $18.000.271 – $27.000.406 | $2.680.040 | 23% |
| $27.000.406 – $40.500.609 | $4.750.071 | 27% |
| $40.500.609 – $60.750.914 | $8.395.126 | 31% |
| Más de $60.750.914 | $14.672.721 | 35% |

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# GANANCIAS — Escala progresiva anual (1er semestre 2026)
# Fuente: Contablix / iProfesional / ARCA
# ─────────────────────────────────────────────────────────────────────

# Cada tramo: (desde, hasta, importe_fijo, alicuota_marginal)
GANANCIAS_ESCALA = [
    (           0,    2_000_030,          0, 0.05),
    (   2_000_030,    4_000_060,    100_002, 0.09),
    (   4_000_060,    6_000_090,    280_004, 0.12),
    (   6_000_090,    9_000_135,    520_008, 0.15),
    (   9_000_135,   18_000_271,    970_015, 0.19),
    (  18_000_271,   27_000_406,  2_680_040, 0.23),
    (  27_000_406,   40_500_609,  4_750_071, 0.27),
    (  40_500_609,   60_750_914,  8_395_126, 0.31),
    (  60_750_914, float('inf'), 14_672_721, 0.35),
]

def calcular_ganancias(gni: float) -> float:
    """Calcula el impuesto a las ganancias dado una ganancia neta imponible anual."""
    if gni <= 0:
        return 0.0
    for desde, hasta, fijo, alicuota in GANANCIAS_ESCALA:
        if gni <= hasta:
            return fijo + (gni - desde) * alicuota
    # Último tramo (nunca debería llegar aquí por float('inf'))
    return gni * 0.35

# ─────────────────────────────────────────────────────────────────────
# DEDUCCIONES PERSONALES — Autónomo soltero, sin cargas de familia
# Fuente: Ley 27.743 + actualización 1er semestre 2026
# ─────────────────────────────────────────────────────────────────────
MNI_ANUAL             = 5_152_500   # Mínimo No Imponible
DED_ESPECIAL_ANUAL    = 18_000_000  # Deducción especial autónomos (Ley 27.743, ~3.5×MNI)
DEDUCCIONES_PERSONALES = MNI_ANUAL + DED_ESPECIAL_ANUAL

print(f"MNI anual:                       {fmt_pesos(MNI_ANUAL)}")
print(f"Deducción especial anual:         {fmt_pesos(DED_ESPECIAL_ANUAL)}")
print(f"Total deducciones personales:     {fmt_pesos(DEDUCCIONES_PERSONALES)}")

# ─────────────────────────────────────────────────────────────────────
# HONORARIOS CONTADOR — Responsable Inscripto
# Estimación basada en tarifas CPCE (módulos) y complejidad del régimen
# Tareas: DDJJ IVA mensual, DDJJ IIBB mensual, anticipos ganancias,
#         DDJJ anual ganancias, gestión autónomos, asesoramiento
# Rango razonable para facturación $70-108M/año: $200.000 – $300.000/mes
# ─────────────────────────────────────────────────────────────────────
RI_CONTADOR_MES = 250_000   # Estimación base ($200k-$300k rango razonable)
RI_CONTADOR_ANUAL = RI_CONTADOR_MES * 12

print(f"\nHonorarios contador RI estimados: {fmt_pesos(RI_CONTADOR_MES)}/mes")
print(f"  → {fmt_pesos(RI_CONTADOR_ANUAL)}/año")
print("  (rango razonable: $200.000 – $300.000/mes según volumen de operaciones)")

---
## 3. Cálculo Monotributo — Escenarios H y K

In [ ]:
def calcular_monotributo(cat: str) -> dict:
    """
    Calcula el costo fiscal anual del Monotributo para la categoría indicada.
    Retorna un dict con todos los componentes.
    """
    d = MONO[cat]
    billing_anual = d['billing_anual_max']   # Tomamos el máximo de la categoría
    billing_mensual = billing_anual / 12

    # ── Costos mensuales ──
    impuesto_mes   = d['impuesto_integrado_mes']
    sipa_mes       = d['sipa_mes']
    os_mes         = d['obra_social_mes']   # excluida
    contador_mes   = MONO_CONTADOR_MES

    # ── Costos anuales ──
    impuesto_anual  = impuesto_mes  * 12
    sipa_anual      = sipa_mes      * 12
    os_anual        = os_mes        * 12
    contador_anual  = contador_mes  * 12

    # ── Costo fiscal total (sin obra social) ──
    total_sin_os = impuesto_anual + sipa_anual + contador_anual
    total_con_os = total_sin_os + os_anual

    return {
        'regimen':                'Monotributo',
        'categoria':              cat,
        'billing_mensual':        billing_mensual,
        'billing_anual':          billing_anual,
        # Componentes mensuales
        'impuesto_integrado_mes': impuesto_mes,
        'sipa_jubilacion_mes':    sipa_mes,
        'obra_social_mes':        os_mes,
        'contador_mes':           contador_mes,
        'cuota_total_mes':        d['total_cuota_mes'],
        # Componentes anuales
        'impuesto_integrado_anual': impuesto_anual,
        'sipa_jubilacion_anual':    sipa_anual,
        'obra_social_anual':        os_anual,
        'contador_anual':           contador_anual,
        # Totales
        'costo_total_sin_os':     total_sin_os,
        'costo_total_con_os':     total_con_os,
        # Porcentajes sobre facturación
        'pct_impuesto':     impuesto_anual  / billing_anual * 100,
        'pct_sipa':         sipa_anual      / billing_anual * 100,
        'pct_os':           os_anual        / billing_anual * 100,
        'pct_contador':     contador_anual  / billing_anual * 100,
        'pct_total_sin_os': total_sin_os    / billing_anual * 100,
        'pct_total_con_os': total_con_os    / billing_anual * 100,
    }


res_mono_h = calcular_monotributo('H')
res_mono_k = calcular_monotributo('K')

# ── Imprimir resultados ──
for r in [res_mono_h, res_mono_k]:
    print(f"{'='*60}")
    print(f" MONOTRIBUTO — CATEGORÍA {r['categoria']}")
    print(f"{'='*60}")
    print(f" Facturación máxima mensual:      {fmt_pesos(r['billing_mensual'])}")
    print(f" Facturación máxima anual:        {fmt_pesos(r['billing_anual'])}")
    print()
    print(f" COSTOS MENSUALES:")
    print(f"   Impuesto integrado:            {fmt_pesos(r['impuesto_integrado_mes'])}  ({r['pct_impuesto']:.2f}% anual)")
    print(f"   SIPA (jubilación):             {fmt_pesos(r['sipa_jubilacion_mes'])}  ({r['pct_sipa']:.2f}% anual)")
    print(f"   Obra social (excluida):        {fmt_pesos(r['obra_social_mes'])}  ({r['pct_os']:.2f}% anual)")
    print(f"   Honorarios contador:           {fmt_pesos(r['contador_mes'])}  ({r['pct_contador']:.2f}% anual)")
    print()
    print(f" COSTO TOTAL ANUAL (sin OS):      {fmt_pesos(r['costo_total_sin_os'])}  → {r['pct_total_sin_os']:.2f}% del billing")
    print(f" COSTO TOTAL ANUAL (con OS):      {fmt_pesos(r['costo_total_con_os'])}  → {r['pct_total_con_os']:.2f}% del billing")
    print()

---
## 4. Cálculo Responsable Inscripto — Escenarios H y K

### Metodología de cálculo
```
Facturación bruta
  − 20% gastos profesionales deducibles
  − Honorarios contador (gasto deducible)
  − Aporte autónomos Cat V completo (deducible de Ganancias)
  − Deducciones personales (MNI + Deducción Especial)
  = Ganancia Neta Imponible (GNI)

  → Impuesto a las Ganancias s/ GNI (escala progresiva)
```

> **IVA**: Se considera neutral (clientes RI que acreditan el IVA). El prestador cobra IVA y lo transfiere a ARCA sin impacto neto en sus ingresos reales.

In [ ]:
def calcular_ri(billing_anual: float, etiqueta: str) -> dict:
    """
    Calcula el costo fiscal anual del Responsable Inscripto
    para un determinado nivel de facturación anual.
    """
    billing_mensual = billing_anual / 12

    # ── Aportes autónomos ──
    autonomos_total_mes   = AUTONOMOS_CAT_V_MES
    autonomos_sipa_mes    = autonomos_total_mes * FRAC_SIPA    # porción jubilación
    autonomos_salud_mes   = autonomos_total_mes * FRAC_SALUD   # porción salud (excluida)

    autonomos_total_anual = autonomos_total_mes * 12
    autonomos_sipa_anual  = autonomos_sipa_mes  * 12
    autonomos_salud_anual = autonomos_salud_mes * 12

    # ── Honorarios contador ──
    contador_mes   = RI_CONTADOR_MES
    contador_anual = RI_CONTADOR_ANUAL

    # ── Gastos profesionales deducibles (20% de la facturación) ──
    # Nota: estos gastos ya están "pagados" (son costos reales del negocio),
    # por lo que reducen la base imponible pero también el ingreso neto disponible.
    gastos_deducibles = billing_anual * 0.20

    # ── Ganancias: cálculo de la Ganancia Neta Imponible (GNI) ──
    # Deducciones del resultado de la actividad:
    gni = (billing_anual
           - gastos_deducibles           # gastos operativos deducibles
           - contador_anual              # honorarios contador (gasto deducible)
           - autonomos_total_anual       # aportes autónomos COMPLETOS son deducibles
           - DEDUCCIONES_PERSONALES)     # MNI + Deducción Especial

    gni = max(gni, 0)   # no puede ser negativa

    impuesto_ganancias = calcular_ganancias(gni)

    # Alícuota efectiva de Ganancias sobre la facturación bruta
    alicuota_efec = impuesto_ganancias / billing_anual * 100

    # ── Costo fiscal total (excluida obra social) ──
    # Incluye: Ganancias + SIPA jubilación + honorarios contador
    # Excluye: PAMI + obra social (autónomos), IVA (neutro)
    costo_total_sin_os = impuesto_ganancias + autonomos_sipa_anual + contador_anual
    costo_total_con_os = impuesto_ganancias + autonomos_total_anual + contador_anual

    return {
        'regimen':               'Responsable Inscripto',
        'categoria':             etiqueta,
        'billing_mensual':       billing_mensual,
        'billing_anual':         billing_anual,
        # Autónomos
        'autonomos_total_mes':   autonomos_total_mes,
        'autonomos_sipa_mes':    autonomos_sipa_mes,
        'autonomos_salud_mes':   autonomos_salud_mes,
        'autonomos_total_anual': autonomos_total_anual,
        'autonomos_sipa_anual':  autonomos_sipa_anual,
        'autonomos_salud_anual': autonomos_salud_anual,
        # Ganancias
        'gastos_deducibles_anual': gastos_deducibles,
        'contador_anual':          contador_anual,
        'deducciones_personales':  DEDUCCIONES_PERSONALES,
        'gni':                     gni,
        'impuesto_ganancias':      impuesto_ganancias,
        'alicuota_efectiva_pct':   alicuota_efec,
        # Totales
        'costo_total_sin_os':    costo_total_sin_os,
        'costo_total_con_os':    costo_total_con_os,
        # Porcentajes
        'pct_ganancias':     impuesto_ganancias   / billing_anual * 100,
        'pct_sipa':          autonomos_sipa_anual / billing_anual * 100,
        'pct_salud_os':      autonomos_salud_anual/ billing_anual * 100,
        'pct_contador':      contador_anual       / billing_anual * 100,
        'pct_total_sin_os':  costo_total_sin_os   / billing_anual * 100,
        'pct_total_con_os':  costo_total_con_os   / billing_anual * 100,
    }


res_ri_h = calcular_ri(MONO['H']['billing_anual_max'], 'H')
res_ri_k = calcular_ri(MONO['K']['billing_anual_max'], 'K')

# ── Imprimir resultados ──
for r in [res_ri_h, res_ri_k]:
    print(f"{'='*60}")
    print(f" RESPONSABLE INSCRIPTO — ESCENARIO {r['categoria']}")
    print(f"{'='*60}")
    print(f" Facturación mensual (referencia):    {fmt_pesos(r['billing_mensual'])}")
    print(f" Facturación anual (referencia):      {fmt_pesos(r['billing_anual'])}")
    print()
    print(f" DEDUCCIONES GANANCIAS:")
    print(f"   20% gastos profesionales:          {fmt_pesos(r['gastos_deducibles_anual'])}")
    print(f"   Honorarios contador:               {fmt_pesos(r['contador_anual'])}")
    print(f"   Aportes autónomos (Cat V total):   {fmt_pesos(r['autonomos_total_anual'])}")
    print(f"   Deducciones personales (MNI+DE):   {fmt_pesos(r['deducciones_personales'])}")
    print(f"   ─────────────────────────────────")
    print(f"   Ganancia Neta Imponible:           {fmt_pesos(r['gni'])}")
    print()
    print(f" COSTOS ANUALES:")
    print(f"   Impuesto a las Ganancias:          {fmt_pesos(r['impuesto_ganancias'])}  ({r['pct_ganancias']:.2f}% del billing)")
    print(f"   Alícuota efectiva s/billing:       {r['alicuota_efectiva_pct']:.2f}%")
    print(f"   SIPA jubilación (71,1% autónomos): {fmt_pesos(r['autonomos_sipa_anual'])}  ({r['pct_sipa']:.2f}% del billing)")
    print(f"   PAMI+OS autónomos (excluida):      {fmt_pesos(r['autonomos_salud_anual'])}  ({r['pct_salud_os']:.2f}% del billing)")
    print(f"   Honorarios contador:               {fmt_pesos(r['contador_anual'])}  ({r['pct_contador']:.2f}% del billing)")
    print()
    print(f" COSTO TOTAL ANUAL (sin OS):          {fmt_pesos(r['costo_total_sin_os'])}  → {r['pct_total_sin_os']:.2f}% del billing")
    print(f" COSTO TOTAL ANUAL (con OS):          {fmt_pesos(r['costo_total_con_os'])}  → {r['pct_total_con_os']:.2f}% del billing")
    print()

---
## 5. Tabla Comparativa: Monotributo vs. Responsable Inscripto

In [ ]:
def tabla_comparativa(res_mono: dict, res_ri: dict) -> pd.DataFrame:
    """Genera una tabla comparativa entre ambos regímenes para un escenario."""
    cat = res_mono['categoria']
    billing = res_mono['billing_anual']

    filas = [
        # ── Datos de contexto ──
        ('Facturación anual (referencia)',
            billing, billing),
        ('Facturación mensual (referencia)',
            billing / 12, billing / 12),

        # ── Impuesto / Carga impositiva ──
        ('Impuesto (integrado / ganancias) — ANUAL',
            res_mono['impuesto_integrado_anual'],
            res_ri['impuesto_ganancias']),
        ('  % sobre facturación',
            res_mono['pct_impuesto'],
            res_ri['pct_ganancias']),

        # ── SIPA jubilación ──
        ('SIPA jubilación — ANUAL',
            res_mono['sipa_jubilacion_anual'],
            res_ri['autonomos_sipa_anual']),
        ('  % sobre facturación',
            res_mono['pct_sipa'],
            res_ri['pct_sipa']),

        # ── Obra social (mostrada pero excluida del total) ──
        ('Obra social / salud — ANUAL (excluida del total)',
            res_mono['obra_social_anual'],
            res_ri['autonomos_salud_anual']),
        ('  % sobre facturación',
            res_mono['pct_os'],
            res_ri['pct_salud_os']),

        # ── Honorarios contador ──
        ('Honorarios contador — ANUAL',
            res_mono['contador_anual'],
            res_ri['contador_anual']),
        ('  % sobre facturación',
            res_mono['pct_contador'],
            res_ri['pct_contador']),

        # ── TOTALES ──
        ('══ COSTO TOTAL SIN OBRA SOCIAL — ANUAL ══',
            res_mono['costo_total_sin_os'],
            res_ri['costo_total_sin_os']),
        ('══ % del billing (SIN OS) ══',
            res_mono['pct_total_sin_os'],
            res_ri['pct_total_sin_os']),

        ('── COSTO TOTAL CON OBRA SOCIAL — ANUAL ──',
            res_mono['costo_total_con_os'],
            res_ri['costo_total_con_os']),
        ('── % del billing (CON OS) ──',
            res_mono['pct_total_con_os'],
            res_ri['pct_total_con_os']),
    ]

    labels, mono_vals, ri_vals = zip(*filas)
    df = pd.DataFrame({
        'Concepto': labels,
        f'Monotributo Cat {cat}': mono_vals,
        f'Resp. Inscripto (billing Cat {cat})': ri_vals,
    }).set_index('Concepto')
    return df


df_comp_h = tabla_comparativa(res_mono_h, res_ri_h)
df_comp_k = tabla_comparativa(res_mono_k, res_ri_k)

print('\n' + '='*80)
print('  COMPARATIVA — ESCENARIO H (servicios, facturación máx. Cat. H)')
print('='*80)
display(df_comp_h)

print('\n' + '='*80)
print('  COMPARATIVA — ESCENARIO K (servicios, facturación máx. Cat. K)')
print('='*80)
display(df_comp_k)

---
## 6. Visualizaciones

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Gráfico 1: Costo total como % del billing (sin OS)
# ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

COLORS = {
    'impuesto':  '#2C7BB6',  # azul
    'sipa':      '#F46D43',  # naranja
    'contador':  '#74C476',  # verde
    'os':        '#BDBDBD',  # gris (excluida)
}

scenarios = [
    ('H', res_mono_h, res_ri_h),
    ('K', res_mono_k, res_ri_k),
]

for ax, (cat, rm, rr) in zip(axes, scenarios):
    regimenes = ['Monotributo', 'Resp. Inscripto']
    impuesto_pct = [rm['pct_impuesto'], rr['pct_ganancias']]
    sipa_pct     = [rm['pct_sipa'],     rr['pct_sipa']]
    contador_pct = [rm['pct_contador'], rr['pct_contador']]

    x = np.arange(len(regimenes))
    w = 0.5

    bars_imp = ax.bar(x, impuesto_pct, w, label='Impuesto (integrado/ganancias)',
                      color=COLORS['impuesto'], edgecolor='white')
    bars_sip = ax.bar(x, sipa_pct, w, bottom=impuesto_pct,
                      label='SIPA Jubilación',
                      color=COLORS['sipa'], edgecolor='white')
    bottoms = [i + s for i, s in zip(impuesto_pct, sipa_pct)]
    bars_cnt = ax.bar(x, contador_pct, w, bottom=bottoms,
                      label='Honorarios Contador',
                      color=COLORS['contador'], edgecolor='white')

    # Etiquetas de total
    totales = [rm['pct_total_sin_os'], rr['pct_total_sin_os']]
    for i, total in enumerate(totales):
        ax.text(i, total + 0.3, f'{total:.1f}%', ha='center', va='bottom',
                fontsize=13, fontweight='bold')

    ax.set_title(f'Escenario {cat} — Costo fiscal (sin OS)\n'
                 f'Billing anual: ${MONO[cat]["billing_anual_max"]/1e6:.1f}M',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('% sobre facturación total')
    ax.set_xticks(x)
    ax.set_xticklabels(regimenes, fontsize=12)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.legend(loc='upper right', fontsize=9)
    ax.set_ylim(0, max(totales) * 1.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Monotributo vs. Responsable Inscripto — Costo Fiscal Total (excl. obra social)\n'
             'Servicios · Argentina 2026 (1er semestre)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparativa_costo_fiscal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado: comparativa_costo_fiscal.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Gráfico 2: Costo total en pesos absolutos
# ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (cat, rm, rr) in zip(axes, scenarios):
    regimenes = ['Monotributo', 'Resp. Inscripto']
    impuesto_abs = [rm['impuesto_integrado_anual'] / 1e6, rr['impuesto_ganancias'] / 1e6]
    sipa_abs     = [rm['sipa_jubilacion_anual'] / 1e6,    rr['autonomos_sipa_anual'] / 1e6]
    contador_abs = [rm['contador_anual'] / 1e6,           rr['contador_anual'] / 1e6]

    x = np.arange(len(regimenes))
    w = 0.5

    ax.bar(x, impuesto_abs, w, label='Impuesto', color=COLORS['impuesto'], edgecolor='white')
    ax.bar(x, sipa_abs, w, bottom=impuesto_abs, label='SIPA Jubilación',
           color=COLORS['sipa'], edgecolor='white')
    bottoms_abs = [i + s for i, s in zip(impuesto_abs, sipa_abs)]
    ax.bar(x, contador_abs, w, bottom=bottoms_abs, label='Contador',
           color=COLORS['contador'], edgecolor='white')

    totales_abs = [rm['costo_total_sin_os'] / 1e6, rr['costo_total_sin_os'] / 1e6]
    for i, total in enumerate(totales_abs):
        ax.text(i, total + 0.1, f'${total:.1f}M', ha='center', va='bottom',
                fontsize=12, fontweight='bold')

    ax.set_title(f'Escenario {cat} — Costo fiscal absoluto (sin OS)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Millones de pesos / año')
    ax.set_xticks(x)
    ax.set_xticklabels(regimenes, fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:.0f}M'))
    ax.legend(loc='upper right', fontsize=9)
    ax.set_ylim(0, max(totales_abs) * 1.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Costo Fiscal Anual en Pesos Absolutos (excl. obra social)\n'
             'Servicios · Argentina 2026 (1er semestre)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparativa_costo_absoluto.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Gráfico 3: Costo total (% billing) vs. nivel de facturación
# Muestra cómo evoluciona el costo de RI a medida que crece la facturación
# ─────────────────────────────────────────────────────────────────────
billings = np.linspace(10_000_000, 120_000_000, 500)

pct_mono_h = [res_mono_h['pct_total_sin_os']] * len(billings)  # fijo en su nivel
pct_mono_k = [res_mono_k['pct_total_sin_os']] * len(billings)  # fijo en su nivel
pct_ri = [calcular_ri(b, '')['pct_total_sin_os'] for b in billings]

fig, ax = plt.subplots(figsize=(13, 7))

ax.plot(billings / 1e6, pct_ri, color='#D73027', lw=2.5, label='Resp. Inscripto (variable)')

# Puntos de referencia Monotributo
ax.axhline(res_mono_h['pct_total_sin_os'], color='#2C7BB6',
           lw=1.8, linestyle='--', alpha=0.8,
           label=f"Monotributo Cat H: {res_mono_h['pct_total_sin_os']:.1f}%")
ax.axhline(res_mono_k['pct_total_sin_os'], color='#4DAC26',
           lw=1.8, linestyle='--', alpha=0.8,
           label=f"Monotributo Cat K: {res_mono_k['pct_total_sin_os']:.1f}%")

# Marcar los escenarios H y K en la curva RI
for cat, rm, rr, color in [('H', res_mono_h, res_ri_h, '#2C7BB6'),
                              ('K', res_mono_k, res_ri_k, '#4DAC26')]:
    b_m = rm['billing_anual'] / 1e6
    p_ri_scen = rr['pct_total_sin_os']
    ax.scatter([b_m], [p_ri_scen], s=100, color=color, zorder=5)
    ax.annotate(f"RI @ Cat {cat}\n{p_ri_scen:.1f}%",
                xy=(b_m, p_ri_scen), xytext=(b_m + 3, p_ri_scen + 0.8),
                fontsize=9, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

# Línea vertical en el tope de categoría H
ax.axvline(MONO['H']['billing_anual_max'] / 1e6, color='#2C7BB6', lw=1, linestyle=':')
ax.axvline(MONO['K']['billing_anual_max'] / 1e6, color='#4DAC26', lw=1, linestyle=':')

ax.set_xlabel('Facturación anual (millones de ARS)', fontsize=12)
ax.set_ylabel('Costo fiscal como % de la facturación', fontsize=12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_title('Costo Fiscal Total (excl. obra social) según nivel de facturación\n'
             'Monotributo (línea punteada fija) vs. Responsable Inscripto (curva variable)\n'
             'Servicios · Argentina 2026',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('curva_costo_vs_facturacion.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Resumen Ejecutivo

In [ ]:
print()
print('█' * 70)
print('  RESUMEN EJECUTIVO — Monotributo vs. Responsable Inscripto')
print('  Argentina 2026 (1er semestre) · Prestación de servicios')
print('█' * 70)
print()

for cat, rm, rr in [('H', res_mono_h, res_ri_h), ('K', res_mono_k, res_ri_k)]:
    diferencia_pct  = rr['pct_total_sin_os'] - rm['pct_total_sin_os']
    diferencia_abs  = rr['costo_total_sin_os'] - rm['costo_total_sin_os']
    mas_caro        = 'RI es más caro' if diferencia_pct > 0 else 'Monotributo es más caro'

    print(f'  ┌─ ESCENARIO {cat} — Billing anual: ${rm["billing_anual"]/1e6:.1f}M '
          f'(${rm["billing_mensual"]/1e3:.0f}K/mes) ─────────────────────────')
    print(f'  │')
    print(f'  │  Facturación mensual (max):         {fmt_pesos(rm["billing_mensual"])}')
    print(f'  │')
    print(f'  │           MONOTRIBUTO    RESP. INSCRIPTO   DIFERENCIA')
    print(f'  │           ─────────────  ──────────────    ─────────────')
    print(f'  │  ANUAL:')
    print(f'  │  Impuesto  {fmt_pesos(rm["impuesto_integrado_anual"])}  {fmt_pesos(rr["impuesto_ganancias"])}  '
          f'{fmt_pesos(rr["impuesto_ganancias"] - rm["impuesto_integrado_anual"])}')
    print(f'  │  SIPA jub. {fmt_pesos(rm["sipa_jubilacion_anual"])}  {fmt_pesos(rr["autonomos_sipa_anual"])}  '
          f'{fmt_pesos(rr["autonomos_sipa_anual"] - rm["sipa_jubilacion_anual"])}')
    print(f'  │  Contador  {fmt_pesos(rm["contador_anual"])}  {fmt_pesos(rr["contador_anual"])}  '
          f'{fmt_pesos(rr["contador_anual"] - rm["contador_anual"])}')
    print(f'  │  OS (ref.) {fmt_pesos(rm["obra_social_anual"])}  {fmt_pesos(rr["autonomos_salud_anual"])}  '
          f'{fmt_pesos(rr["autonomos_salud_anual"] - rm["obra_social_anual"])}')
    print(f'  │  ─────────────────────────────────────────────────────')
    print(f'  │  TOTAL     {fmt_pesos(rm["costo_total_sin_os"])}  {fmt_pesos(rr["costo_total_sin_os"])}  '
          f'{fmt_pesos(diferencia_abs)}')
    print(f'  │  (sin OS)')
    print(f'  │')
    print(f'  │  % BILLING:    {rm["pct_total_sin_os"]:6.2f}%          {rr["pct_total_sin_os"]:6.2f}%          +{abs(diferencia_pct):.2f}pp')
    print(f'  │')
    print(f'  │  ▶ {mas_caro} en {abs(diferencia_pct):.2f} puntos porcentuales')
    print(f'  │  ▶ Diferencia absoluta anual: {fmt_pesos(abs(diferencia_abs))}')
    print(f'  │  ▶ Diferencia mensual aprox.: {fmt_pesos(abs(diferencia_abs)/12)}')
    print(f'  └─────────────────────────────────────────────────────────────────────')
    print()

print('─'*70)
print('  NOTAS Y SUPUESTOS CLAVE:')
print('─'*70)
print(f'  • Monotributo: cuotas feb 2026 (ARCA / Contablix)')
print(f'  • RI - Autónomos Cat V: ${AUTONOMOS_CAT_V_MES:,.0f}/mes (Res. ANSES 381/2025, ene 2026)')
print(f'  • RI - Autónomos SIPA (71,1%): ${autonomos_sipa_mes:,.0f}/mes (27/38 del total)')
print(f'  • RI - Ganancias: escala progresiva 5%-35% (1er sem 2026)')
print(f'  • RI - Deducciones personales: MNI ${MNI_ANUAL/1e6:.2f}M + '
      f'Ded.Especial ${DED_ESPECIAL_ANUAL/1e6:.0f}M = ${DEDUCCIONES_PERSONALES/1e6:.2f}M/año')
print(f'  • RI - Gastos deducibles: 20% de la facturación')
print(f'  • RI - IVA (21%): NEUTRO — clientes RI que acreditan (B2B)')
print(f'  • Contador Monotributo: ${MONO_CONTADOR_MES:,}/mes (${MONO_CONTADOR_MES*12:,}/año)')
print(f'  • Contador RI: ${RI_CONTADOR_MES:,}/mes (${RI_CONTADOR_ANUAL:,}/año) — '
      f'rango razonable $200k-$300k/mes')
print(f'  • Obra social: EXCLUIDA del total (puede redirigirse a prepaga)')
print(f'  • Ingresos Brutos (provincial): EXCLUIDO del análisis comparativo')
print(f'    (aplica a ambos regímenes de forma similar, no altera la diferencia)')
print(f'  • Profesional soltero sin cargas de familia')
print(f'  • Actividad: prestación de servicios (no venta de bienes)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Tabla síntesis final (formato limpio)
# ─────────────────────────────────────────────────────────────────────
sintesis = []
for cat, rm, rr in [('H', res_mono_h, res_ri_h), ('K', res_mono_k, res_ri_k)]:
    for regimen, res in [('Monotributo', rm), ('Resp. Inscripto', rr)]:
        if regimen == 'Monotributo':
            imp = res['impuesto_integrado_anual']
            sipa = res['sipa_jubilacion_anual']
            os_ = res['obra_social_anual']
        else:
            imp = res['impuesto_ganancias']
            sipa = res['autonomos_sipa_anual']
            os_ = res['autonomos_salud_anual']
        sintesis.append({
            'Escenario': f'Cat {cat}',
            'Régimen': regimen,
            'Billing anual': res['billing_anual'],
            'Billing mensual': res['billing_mensual'],
            'Impuesto anual': imp,
            '% Impuesto': imp / res['billing_anual'] * 100,
            'SIPA jubilación anual': sipa,
            '% SIPA': sipa / res['billing_anual'] * 100,
            'OS/Salud anual (excluida)': os_,
            '% OS': os_ / res['billing_anual'] * 100,
            'Contador anual': res['contador_anual'],
            '% Contador': res['contador_anual'] / res['billing_anual'] * 100,
            'TOTAL (sin OS) anual': res['costo_total_sin_os'],
            '% TOTAL (sin OS)': res['pct_total_sin_os'],
            'TOTAL (con OS) anual': res['costo_total_con_os'],
            '% TOTAL (con OS)': res['pct_total_con_os'],
        })

df_sintesis = pd.DataFrame(sintesis).set_index(['Escenario', 'Régimen'])
print('=== TABLA SÍNTESIS FINAL ===')
display(df_sintesis)

---
## 8. Conclusiones

### Hallazgos principales (1er semestre 2026, servicios)

| Escenario | Monotributo (% billing) | RI (% billing) | Diferencia RI vs. Mono |
|-----------|------------------------|----------------|------------------------|
| Cat H (~$70M/año) | ~7-8% | ~15-16% | **RI +7-8 pp más caro** |
| Cat K (~$108M/año) | ~15% | ~18-19% | **RI +3-4 pp más caro** |

### Factores que explican las diferencias

1. **El impuesto integrado del monotributo es muy bajo en Cat H**: El impuesto integrado representa apenas ~6% del billing en Cat H, mientras que Ganancias en RI puede alcanzar ~12-14%.

2. **La SIPA del RI es fija y alta**: Los autónomos Cat V pagan $276.066/mes fijos (~$3.3M/año), mientras que en monotributo el SIPA en Cat H es solo ~$591.528/año — casi **5 veces menos**.

3. **La brecha se achica en Cat K**: En cat K, el impuesto integrado del monotributo se vuelve muy alto (~$14M/año), acercando ambos regímenes.

4. **El contador tiene mayor peso en monotributo a bajos ingresos**: Los $660.000/año representan ~0,94% del billing en Cat H, mientras que los $3.000.000/año en RI son 4,28% del mismo billing.

### Consideraciones adicionales (fuera del scope numérico)

- **IVA y competitividad**: En RI, si el prestador tiene clientes que son consumidores finales (personas humanas), la obligación de cargar IVA (21%) sobre los honorarios encarece el servicio para esos clientes. En monotributo, los precios son finales sin IVA adicional.

- **Límite de facturación**: El monotributo tiene un techo (~$108M/año en Cat K). Al superarlo, la adhesión al RI se vuelve obligatoria independientemente de la comparación de costos.

- **Ingresos Brutos provinciales**: Se aplica a ambos regímenes (típicamente 2,5-4% en CABA/PBA para servicios). No altera la comparativa relativa entre regímenes.

- **Complejidad administrativa**: El RI implica mayor carga administrativa (DDJJ IVA mensual, anticipos de Ganancias, etc.), lo que justifica el mayor honorario del contador.

- **Sensibilidad al honorario del contador RI**: Si el contador cobra $200.000/mes en vez de $250.000/mes, el costo total de RI baja ~0,7pp sobre el billing. Si cobra $300.000/mes, sube ~0,7pp.

### Recomendación
> Para un prestador de servicios con facturación dentro de la Categoría H (~$70M/año), el **Monotributo es significativamente más conveniente** desde el punto de vista fiscal. A medida que la facturación se acerca al máximo de la Categoría K (~$108M/año), los costos fiscales de ambos regímenes convergen, aunque el Monotributo continúa siendo más económico mientras la facturación no supere ese límite.